### MultiVectorRetriever

In [43]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_text_splitters import HTMLSectionSplitter # for splitting the HTML content
import os
from dotenv import load_dotenv
load_dotenv()
import uuid
from langchain_community.document_transformers import Html2TextTransformer


In [44]:
embeddings = NVIDIAEmbeddings(
        model="nvidia/nv-embedqa-e5-v5",
        api_key=os.getenv("NVIDIA_NV_API"),
)

In [45]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
child_chunks_collection = Chroma(collection_name="uk_child_chunks", embedding_function=embeddings)
child_chunks_collection.reset_collection()


In [46]:
doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store,
    id_key=doc_key,
)


In [47]:
uk_destinations = [
 "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
 "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
 "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
 "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
 "Rye_(England)", "Seaford", "Ashdown_Forest"
]

In [48]:
base_wiki_url = "https://en.wikipedia.org/wiki/"
html2text_transformer = Html2TextTransformer()

for place in uk_destinations:
    destination_url = base_wiki_url + place
    try:
        html_docs = AsyncHtmlLoader(destination_url).load()
        text_docs = html2text_transformer.transform_documents(html_docs)
        parent_docs = parent_splitter.split_documents(text_docs)

        for pdoc in parent_docs:
            pid = str(uuid.uuid4())
            pdoc.metadata[doc_key] = pid

            child_docs = child_splitter.split_documents([pdoc])
            for cdoc in child_docs:
                cdoc.metadata[doc_key] = pid

            child_chunks_collection.add_documents(child_docs)
            multi_vector_retriever.docstore.mset([(pid, pdoc)])

        print(f"Indexed: {destination_url}")
    except Exception as e:
        print(f"Skipped {destination_url}: {e}")


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.70it/s]


Indexed: https://en.wikipedia.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.64it/s]


Indexed: https://en.wikipedia.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.50it/s]


...

Indexed: https://en.wikipedia.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.88it/s]


Indexed: https://en.wikipedia.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.53it/s]


Indexed: https://en.wikipedia.org/wiki/Ashdown_Forest


In [49]:
results = multi_vector_retriever.invoke("best places to visit in Cornwall")

for doc in results[:3]:
    print(doc.page_content[:700])
    print("-" * 40)


Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact
bot-traffic@wikimedia.org if you need higher volumes. (30224bb)
----------------------------------------
Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact
bot-traffic@wikimedia.org if you need higher volumes. (30224bb)
----------------------------------------
Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact
bot-traffic@wikimedia.org if you need higher volumes. (30224bb)
----------------------------------------


In [50]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
retrieved_docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Cornwall', 'doc_id': '0737b068-2cd2-49ed-a9c4-f6db78135b19'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/West_Cornwall', 'doc_id': '1ef00098-a4e1-4cdf-b0d7-b32152aa55ab'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Bodmin', 'doc_id': 'c54c41ce-4252-4a40-901c-40e6ef0bd814'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Wadebridge', 'doc_id': 'b9cd1df6-cb98-4cdf-9cc0-0329be4d3260'}, page_content='Please re

In [51]:
cs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
cs_only

[Document(id='c1b6ca3f-f77a-4fad-9c5a-e90cfdb3beaf', metadata={'source': 'https://en.wikipedia.org/wiki/Cornwall', 'doc_id': '0737b068-2cd2-49ed-a9c4-f6db78135b19'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(id='f5ab4dcf-029d-425d-a1e7-aa0dfc34a3de', metadata={'doc_id': '1ef00098-a4e1-4cdf-b0d7-b32152aa55ab', 'source': 'https://en.wikipedia.org/wiki/West_Cornwall'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(id='04239f2c-cc9a-4f1a-8112-97f377dc1a29', metadata={'source': 'https://en.wikipedia.org/wiki/Bodmin', 'doc_id': 'c54c41ce-4252-4a40-901c-40e6ef0bd814'}, page_content='Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact\nbot-traffic@wikimedia.org if you need higher volumes. (30224bb)'),
 Document(id='08ef5